# Flide 사용법: Jev 호환 API로 판단 모델을 띄우고 vLLM·SGLang 대비 속도 주장 확인하기

위에서부터 차례로 실행해 보세요.

### 설치와 환경 확인

In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
pip install -q uv
uv pip install --system flide torchvision
flide --help | head -12

### T4 용 어텐션 타일 패치 런처 만들기 (A10/L4/A100 이상이면 필요 없음)

In [ ]:
%%bash
cat > /content/flide_t4.py <<'E'
import flide.kernels.attention as A
def cfg(device, dtype, head_dim):
    return {"BLOCK_M": 32, "BLOCK_N": 32, "num_warps": 4, "num_stages": 1}
A.config = cfg
from flide.cli import main
main()
E

### OneJev1.6-0.8B 서버 띄우고 포트 확인

In [ ]:
import subprocess, time, urllib.request
subprocess.Popen("nohup python /content/flide_t4.py serve --model OmniJev/OneJev1.6-0.8B --dtype float16 --host 127.0.0.1 > /content/flide.log 2>&1 &", shell=True)
t = time.time()
while time.time() - t < 600:
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=2); break
    except urllib.error.HTTPError:
        break
    except Exception:
        time.sleep(5)
print(open('/content/flide.log').read()[-1200:])

### choice + noul 질문 보내기

In [ ]:
import json, urllib.request
body = {"state": "Customer reports the ceramic mug in order #1042 arrived broken and asks for a refund. Order was placed 3 days ago and the customer has been a member for 5 years.",
 "questions": {"next_action": {"type": "choice", "instructions": "What is the best next action?", "criteria": {"refund": "Refund the order", "replace": "Send a replacement", "escalate": "Escalate to a human agent"}},
 "urgent": {"type": "noul", "instructions": "Does this need an answer within one hour?"}}}
r = urllib.request.Request('http://127.0.0.1:8000/v1/systemone', json.dumps(body).encode(), {'Content-Type': 'application/json'})
out = json.loads(urllib.request.urlopen(r, timeout=120).read())
print(json.dumps(out, indent=1))
print('urgent?', out['answers']['urgent']['noul'] >= 0.5)

### vLLM·SGLang 설치 (SGLang 은 Rust 와 별도 venv 필요)

In [ ]:
%%bash
uv pip install --system vllm==0.33.0
curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
uv venv -p 3.12 /content/sv
. $HOME/.cargo/env; uv pip install --python /content/sv/bin/python 'sglang[all]==0.5.23'

### 벤치마크 스크립트 만들기

In [ ]:
%%bash
mkdir -p /content/outputs
cat > /content/bench.py <<'E'
import sys,json,time,random,statistics,urllib.request
engine,port,model=sys.argv[1],int(sys.argv[2]),sys.argv[3]
REPS=int(sys.argv[4]) if len(sys.argv)>4 else 7
WARM=2
random.seed(7)
names=['Alder','Brix','Corvane','Dunmore','Elsing','Farrow','Gantry','Holloway','Ivers','Jessamy','Kellan','Lowmere','Marrow','Nettle','Orlin','Pellam','Quill','Rowan','Sedge','Tarn']
trades=['salt','wool','copper','timber','glass','honey','tin','silk','barley','amber','slate','flax','cider','iron','pearl','hemp','clay','wax','coal','lime']
out=[];FACTS=[]
for i,n in enumerate(names):
    year=random.choice(range(1100,1900));pop=random.choice(range(2,90))*100;t=trades[i]
    FACTS.append((n,year,pop,t))
    out.append(f'The town of {n} was founded in {year} on the river bank. Its main trade has always been {t}, which merchants carried to the coast by cart and barge. By the last census {n} had {pop} residents, most of whom worked in the {t} guild halls or on the surrounding farms. Travellers describe the market square of {n} as crowded in spring, when the {t} caravans arrive, and quiet in winter, when the roads are closed by snow. The council of {n} meets every second Tuesday to settle disputes about weights, tolls and the upkeep of the old stone bridge, and its minutes are kept in the church tower.')
TEXT='\n\n'.join(out)
def questions(n):
    qs={}
    for i in range(n):
        name,year,pop,t=FACTS[i]
        opts=[t]+random.Random(i).sample([x for x in trades if x!=t],3)
        random.Random(100+i).shuffle(opts)
        qs[f'q{i+1}']={'type':'choice','instructions':f'According to the passage, what is the main trade of the town of {name}?','criteria':{o:o.capitalize() for o in opts}}
    return qs
def post(b):
    r=urllib.request.Request(f'http://127.0.0.1:{port}/v1/systemone',json.dumps(b).encode(),{'Content-Type':'application/json'})
    t=time.perf_counter();raw=urllib.request.urlopen(r,timeout=600).read();return time.perf_counter()-t,json.loads(raw)
res={'engine':engine,'levels':{}}
for n in [1,5,10,20]:
    lat=[];answers=[]
    for rep in range(WARM+REPS):
        b={'state':f'[request {rep:04d}-{n}]\n\n'+TEXT,'questions':questions(n)}
        if model: b['model']=model
        dt,o=post(b)
        if rep>=WARM: lat.append(dt);answers.append({'in_tokens':o['usage']['input_tokens'],'answers':o['answers']})
    res['levels'][n]={'lat':lat,'median':statistics.median(lat),'answers':answers}
    print(engine,n,'median %.3fs'%statistics.median(lat),'in_tokens',answers[0]['in_tokens'],flush=True)
json.dump(res,open(f'/content/outputs/bench_{engine}.json','w'))
E

### Qwen3.5-4B 를 Flide 로 서빙 (T4 메모리 때문에 토큰 한도 축소)

In [ ]:
import subprocess, time, urllib.request
subprocess.run("pkill -f '[f]lide_t4'", shell=True)
subprocess.Popen("nohup python /content/flide_t4.py serve --model Qwen/Qwen3.5-4B --dtype float16 --host 127.0.0.1 --port 8000 --max-request-tokens 6144 --max-branch-tokens 4096 > /content/flide4b.log 2>&1 &", shell=True)
t=time.time()
while time.time()-t<900:
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=2); break
    except urllib.error.HTTPError: break
    except Exception: time.sleep(5)
print(open('/content/flide4b.log').read()[-500:])

### Flide 측정 (질문 1·5·10·20개)

In [ ]:
%%bash
cd /content && python bench.py flide 8000 '' 7

### vLLM 서빙 (프리픽스 캐시 켬)

In [ ]:
import subprocess, time, urllib.request
subprocess.run("pkill -f '[f]lide_t4'", shell=True)
subprocess.Popen("""nohup vllm serve Qwen/Qwen3.5-4B --dtype half --max-model-len 8192 --port 8001 --gpu-memory-utilization 0.80 --enable-prefix-caching --limit-mm-per-prompt '{"image":0,"video":0}' --max-num-seqs 16 --max-num-batched-tokens 4096 --max-cudagraph-capture-size 16 > /content/vllm.log 2>&1 &""", shell=True)
t=time.time()
while time.time()-t<900:
    try:
        urllib.request.urlopen('http://127.0.0.1:8001/v1/models', timeout=2); break
    except Exception: time.sleep(5)
print(open('/content/vllm.log').read()[-300:])

### vLLM 측정 (캐시 켬)

In [ ]:
%%bash
cd /content && python bench.py vllmcache 8001 Qwen/Qwen3.5-4B 7

### SGLang 서빙 (radix 캐시 끔)

In [ ]:
import subprocess, time, urllib.request
subprocess.run("pkill -f '[v]llm serve'", shell=True)
subprocess.Popen("nohup /content/sv/bin/python -m sglang.launch_server --model-path Qwen/Qwen3.5-4B --dtype half --port 8002 --context-length 8192 --mem-fraction-static 0.8 --attention-backend triton --max-running-requests 16 --cuda-graph-max-bs-decode 16 --disable-radix-cache > /content/sglang.log 2>&1 &", shell=True)
t=time.time()
while time.time()-t<900:
    try:
        urllib.request.urlopen('http://127.0.0.1:8002/v1/models', timeout=2); break
    except Exception: time.sleep(5)
print('ready')

### SGLang 측정

In [ ]:
%%bash
cd /content && python bench.py sglang 8002 Qwen/Qwen3.5-4B 7

### 엔진별 선택지 확률 비교 (TV distance, 1위 일치)

In [ ]:
import json, statistics as st
D = {k: json.load(open(f'/content/outputs/bench_{k}.json'))['levels'] for k in ['flide','vllmcache','sglang']}
n = '20'
for other in ['vllmcache','sglang']:
    tv=[]; agree=tot=0
    for rep in range(7):
        A=D['flide'][n]['answers'][rep]['answers']; B=D[other][n]['answers'][rep]['answers']
        for q,v in A.items():
            p=v['probabilities']; r=B[q]['probabilities']
            tv.append(0.5*sum(abs(p[x]-r[x]) for x in p))
            agree += max(p,key=p.get)==max(r,key=r.get); tot += 1
    print('flide vs', other, 'mean TV %.5f max TV %.5f argmax agree %d/%d' % (st.mean(tv), max(tv), agree, tot))